# Lesson 18 Lab — Structured Outputs and Tool Contracts

**Puzzle:** Is valid JSON enough when an application requires a specific schema?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Applications need typed fields, enums, and required properties, not text that merely resembles JSON. Constrained decoding moves part of that contract into token selection, while application validation still owns semantics and side effects.


## 0. Predict before running

1. Predict whether the unconstrained control parses.
2. Check every required field and enum.
3. Name the boundary between generation and tool execution.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab runs native JSON-schema-constrained generation through SamplingParams, parses the text, validates required fields/types locally, and compares it with an unconstrained control.

- JSON syntax and schema conformance are different gates.
- Constrained decoding cannot validate real-world semantics.
- Tool execution must remain downstream of authorization and validation.


## 2. Derive the mechanism

A structured-output backend masks tokens that would make the partial output invalid under a grammar or schema. This reduces syntax retries but does not verify that a tool exists, arguments are safe, or values are factually correct. Tool calling adds parser and chat-template requirements before any executor should act.

### Mechanism at a glance

```mermaid
flowchart LR
  S["JSON schema / grammar"] --> M["allowed-token mask"]
  L["model logits"] --> M
  M --> G["generated JSON text"]
  G --> V["independent schema validation"]
  V --> A["authorization + semantic checks"]
  A --> T["optional tool execution"]
```

### Walk it step by step

1. **Define the contract.** Write required fields, types, enums, and ranges.
2. **Constrain tokens.** Mask continuations that violate the grammar.
3. **Validate independently.** Parse and apply the same schema outside generation.
4. **Authorize side effects.** Never equate a valid argument object with permission to execute.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 18
LESSON_TITLE = 'Structured Outputs and Tool Contracts'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260830
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | ordinary text generation prompted to return JSON |
| Candidate | native structured output constrained by a JSON schema |
| Held constant | model, schema, prompt, sampling, maximum tokens, and GPU |
| Measurements | native success, JSON parse, schema validation, required fields, and control parse |
| Evidence | `native-backend` |

**Experiment:** Generate one schema-constrained object natively, parse it, and apply an independent validator.


## 5. Inspect the experiment code

The schema is embedded in the artifact and validation does not trust the model's claim. No external function is executed by the notebook.

Do not execute until the code matches the frozen table.


In [2]:
from vllm.sampling_params import StructuredOutputsParams
schema={"type":"object","properties":{"component":{"type":"string"},
        "risk":{"type":"string","enum":["low","medium","high"]},"rollback":{"type":"boolean"}},
        "required":["component","risk","rollback"],"additionalProperties":False}
llm=LLM(**base_engine_args(max_model_len=1024)); prompt="Return a risk object for KV cache configuration."
def validate(text):
    try: value=json.loads(text)
    except Exception: return False,False,None
    valid=(isinstance(value,dict) and set(value)==set(schema["required"])
           and isinstance(value.get("component"),str) and value.get("risk") in {"low","medium","high"}
           and isinstance(value.get("rollback"),bool)); return True,bool(valid),value
row={"success":False,"json_parsed":False,"schema_valid":False,"required_fields_present":False,"output_tokens":0,"error":None}
try:
    constraint=StructuredOutputsParams(json=json.dumps(schema)); setting=SamplingParams(temperature=0.0,max_tokens=80,seed=SEED,structured_outputs=constraint)
    item=llm.generate([prompt],setting,use_tqdm=False)[0]; parsed,valid,value=validate(item.outputs[0].text)
    row.update(success=True,json_parsed=parsed,schema_valid=valid,
               required_fields_present=isinstance(value,dict) and all(k in value for k in schema["required"]),
               output_tokens=len(item.outputs[0].token_ids),value=value,text=item.outputs[0].text)
except Exception as exc: row["error"]=f"{type(exc).__name__}: {exc}"
control=llm.generate([prompt],SamplingParams(temperature=0.0,max_tokens=80,seed=SEED),use_tqdm=False)[0]
cparsed,cvalid,cvalue=validate(control.outputs[0].text)
metrics={"schema":schema,"structured":row,"control":{"json_parsed":cparsed,"schema_valid":cvalid,
         "text":control.outputs[0].text,"value":cvalue}}
analysis=(f"Structured success/JSON/schema={row['success']}/{row['json_parsed']}/{row['schema_valid']}; "
          f"unconstrained JSON parsed={cparsed}. Independent semantic/authorization validation remains required.")


INFO 08-13 00:22:25 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260830, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:22:25 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:22:25 [model.py:1883] Using max model len 1024


INFO 08-13 00:22:25 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:22:25 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=652602) 2026-08-13 00:22:36,840 - INFO - autotuner.py:2397 - flashinfer.jit: [Autotuner]: Loaded 0 configs from <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json
(EngineCore pid=652602) 2026-08-13 00:22:36,840 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=652602) 2026-08-13 00:22:36,897 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=652602) 2026-08-13 00:22:36,903 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=652602) INFO 08-13 00:22:37 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=652602) INFO 08-13 00:22:37 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.63 s


(EngineCore pid=652602) WARNING 08-13 00:22:38 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=652602) WARNING 08-13 00:22:38 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=652602) INFO 08-13 00:22:38 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=652602) INFO 08-13 00:22:38 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=652602) INFO 08-13 00:22:38 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:22:38 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Structured success | yes |
| JSON parsed | yes |
| Schema valid | yes |
| Required fields | yes |
| Control JSON parsed | no |
| Output tokens | 28 |


## 7. Explain the result

Structured success/JSON/schema=True/True/True; unconstrained JSON parsed=False. Independent semantic/authorization validation remains required.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 18, "title": 'Structured Outputs and Tool Contracts', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Constrained decoding can establish syntax/schema form; application meaning and tool safety remain independent responsibilities.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 18,
  "title": "Structured Outputs and Tool Contracts",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260830
  },
  "evidence_label": "native-backend",
  "metrics": {
    "schema": {
      "type": "object",
      "properties": {
        "component": {
          "type": "string"
        },
        "risk": {
          "type": "string",
          "enum": [
            "low",
            "medium",
            "high"
          ]
        },
        "rollback": {
          "type": "boolean"
        }
      },
      "required": [
        "component",
        "risk",
        "rollback"
      ],
      "additionalProperties": false
    },
    "structured": {
      "success": true,
      "json_parsed": true,
      "schema_valid": true,
      "required_fields_present": true,
      "o

## 9. Make the bounded decision

> Constrained decoding can establish syntax/schema form; application meaning and tool safety remain independent responsibilities.

**Acceptance/rollback:** Allow structured output into an application only after schema, semantic, authorization, timeout, and side-effect controls pass.

**Failure analysis:** Backend/schema features change across vLLM versions, and a small model may produce schema-valid but useless values. JSON parsing alone misses enum and range constraints.


## 10. Extend the evidence

Add nested schemas, streaming partials, tool-choice policies, adversarial prompts, retries, and a sandboxed mock executor with an audit log.

The full boundary and references are in [`README.md`](README.md).
